# 02 Structured Multi-Tenant RLS with Custom Scopes

## Overview

In this tutorial we'll explore a different approach to multi-tenancy - using Row-Level Security (RLS) in PostgreSQL for structured data. Unlike the previous tutorials that used Knowledge Bases for unstructured documents, this tutorial focuses on querying structured data like insurance claims from a relational database.

The key difference is that we're using Access Tokens (like Tutorial 01) but with a database-level isolation mechanism. Since Access Tokens don't contain custom claims, we need to maintain a user-to-tenant mapping table to determine which tenant's data a user can access.

In this example, we'll:
1. Set up a PostgreSQL database with Row-Level Security
2. Create a Cognito User Pool with Access Token authentication
3. Build a user-to-tenant mapping system
4. Create a Strands Agent that queries the database with RLS
5. Test multi-tenant isolation at the database layer

### Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Structured Data with RLS |
| Authentication | OAuth 2.0 Access Token |
| Agentic Framework | Strands Agents |
| LLM model | Anthropic Claude Sonnet |
| Data store | PostgreSQL with RLS |
| Tenant isolation | Row-Level Security (database layer) |
| Tutorial vertical | Insurance Claims |
| Example complexity | Intermediate |
| SDK used | Amazon Bedrock AgentCore Python SDK |

### Tutorial Architecture

```
Insurance Broker → Access Token → AgentCore → Strands Agent
                   (OAuth 2.0)                      ↓
                                        Lookup tenant from user_sub
                                                     ↓
                                        SET app.current_tenant_id = 'regular'
                                                     ↓
                                        PostgreSQL Query + RLS Policy
                                                     ↓
                                        Returns only tenant's claims
```

### Tutorial Key Features

* PostgreSQL Row-Level Security for tenant isolation
* Access Token authentication (standard OAuth 2.0)
* User-to-tenant mapping table
* Database-level security enforcement
* Structured data queries (SQL)
* Natural language to SQL with Strands Agent

## Prerequisites

To execute this tutorial you will need:
* Python 3.9+
* AWS credentials configured
* Amazon Bedrock access
* Bedrock AgentCore SDK
* Strands Agents
* **PostgreSQL database** (local or RDS)

Note: This tutorial requires a PostgreSQL database. You can use a local PostgreSQL instance or Amazon RDS.

In [ ]:
!pip install -q boto3 bedrock-agentcore-starter-toolkit strands-agents psycopg2-binary pyyaml

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import boto3
import json
import time
import yaml
from datetime import datetime, timedelta
from boto3.session import Session
from utils.aws_resources import CognitoManager, PostgreSQLManager

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

print(f"Region: {region}")
print(f"Account ID: {account_id}")

## Configuration

Let's set up the configuration for this tutorial. You'll need to provide your PostgreSQL connection details. If you don't have a PostgreSQL database yet, you can create one using Amazon RDS or run PostgreSQL locally using Docker.

In [ ]:
PROJECT_NAME = "structured-rls"

user_pool_name = f"{PROJECT_NAME}-pool"

# PostgreSQL configuration
# Update these with your database credentials
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'insurance_claims')
DB_USER = os.getenv('DB_USER', 'postgres')
DB_PASSWORD = os.getenv('DB_PASSWORD', 'postgres')
DB_PORT = os.getenv('DB_PORT', '5432')

print("Configuration:")
print(f"  User Pool: {user_pool_name}")
print(f"  Database: {DB_HOST}:{DB_PORT}/{DB_NAME}")
print(f"\n💡 Tip: Set environment variables DB_HOST, DB_PASSWORD, etc. for your database")

## Setting Up Cognito with Access Tokens

We'll create a Cognito User Pool using standard Access Tokens. Unlike Tutorial 02 where we used ID Tokens with custom claims, Access Tokens don't carry tenant information. This means we need to maintain a separate mapping table in our database to associate users with their tenants.

In [ ]:
cognito = CognitoManager(region)

cognito_config = cognito.get_or_create_user_pool(
    pool_name=user_pool_name,
    schema=[
        {'Name': 'email', 'AttributeDataType': 'String', 'Mutable': True, 'Required': True}
    ],
    auto_verified_attributes=['email']
)

user_pool_id = cognito_config['user_pool_id']
client_id = cognito_config['client_id']
discovery_url = cognito_config['discovery_url']

print(f"✓ User Pool ID: {user_pool_id}")
print(f"✓ Client ID: {client_id}")

### Creating Test Users

Let's create test users for different insurance brokers. Note that we're not adding any custom attributes here - the tenant association will be stored in the database.

In [ ]:
test_users = [
    {
        'username': 'broker-regular',
        'password': 'Pass123!',
        'email': 'broker.regular@example.com',
        'tenant_id': 'regular-insurance'
    },
    {
        'username': 'broker-platinum',
        'password': 'Pass123!',
        'email': 'broker.platinum@example.com',
        'tenant_id': 'platinum-insurance'
    }
]

for user in test_users:
    cognito.get_or_create_user(
        user_pool_id=user_pool_id,
        username=user['username'],
        password=user['password'],
        attributes=[
            {'Name': 'email', 'Value': user['email']},
            {'Name': 'email_verified', 'Value': 'true'}
        ]
    )
    print(f"  Created user: {user['username']}")

print(f"\n✓ {len(test_users)} users ready")

## Setting Up PostgreSQL with Row-Level Security

Now we'll set up our PostgreSQL database with Row-Level Security (RLS). RLS is a powerful feature that automatically filters query results based on the current user context. This provides defense-in-depth - even if there's a bug in the application code, the database ensures users can only see their own data.

In [ ]:
db = PostgreSQLManager(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)

db.connect()

### Creating Database Schema

We'll create three tables:
1. **tenants** - Stores tenant information
2. **user_tenants** - Maps Cognito users to tenants
3. **claims** - Stores insurance claims with RLS enabled

In [ ]:
db.create_schema_with_rls()

## Step 5: Insert Sample Data

In [ ]:
with conn.cursor() as cur:
    # Insert tenants
    cur.execute("""
        INSERT INTO tenants (tenant_id, name) VALUES
        ('broker-a', 'Broker A Insurance'),
        ('broker-b', 'Broker B Insurance')
        ON CONFLICT (tenant_id) DO NOTHING
    """)
    
    # Insert claims for Broker A
    cur.execute("""
        INSERT INTO claims (tenant_id, claim_number, customer_name, claim_amount, claim_date, status) VALUES
        ('broker-a', 'CLM-A-001', 'John Doe', 5000.00, CURRENT_DATE - INTERVAL '2 days', 'approved'),
        ('broker-a', 'CLM-A-002', 'Jane Smith', 3500.00, CURRENT_DATE - INTERVAL '5 days', 'pending'),
        ('broker-a', 'CLM-A-003', 'Bob Johnson', 7200.00, CURRENT_DATE - INTERVAL '1 day', 'approved'),
        ('broker-a', 'CLM-A-004', 'Alice Brown', 4100.00, CURRENT_DATE - INTERVAL '3 days', 'approved'),
        ('broker-a', 'CLM-A-005', 'Charlie Wilson', 2800.00, CURRENT_DATE - INTERVAL '6 days', 'rejected')
    """)
    
    # Insert claims for Broker B
    cur.execute("""
        INSERT INTO claims (tenant_id, claim_number, customer_name, claim_amount, claim_date, status) VALUES
        ('broker-b', 'CLM-B-001', 'David Lee', 6500.00, CURRENT_DATE - INTERVAL '4 days', 'approved'),
        ('broker-b', 'CLM-B-002', 'Emma Davis', 4800.00, CURRENT_DATE - INTERVAL '2 days', 'pending')
    """)
    
conn.commit()
print("✓ Sample data inserted")

## Step 6: Test RLS

In [ ]:
def query_claims(tenant_id):
    """Query claims with tenant context"""
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        # Set tenant context
        cur.execute("SET app.current_tenant_id = %s", (tenant_id,))
        
        # Query claims (RLS automatically filters)
        cur.execute("""
            SELECT claim_number, customer_name, claim_amount, status
            FROM claims
            WHERE claim_date >= CURRENT_DATE - INTERVAL '7 days'
        """)
        
        return cur.fetchall()

# Test Broker A
print("\n=== Broker A Claims ===")
broker_a_claims = query_claims('broker-a')
for claim in broker_a_claims:
    print(f"  {claim['claim_number']}: {claim['customer_name']} - ${claim['claim_amount']} ({claim['status']})")
print(f"Total: {len(broker_a_claims)} claims")

# Test Broker B
print("\n=== Broker B Claims ===")
broker_b_claims = query_claims('broker-b')
for claim in broker_b_claims:
    print(f"  {claim['claim_number']}: {claim['customer_name']} - ${claim['claim_amount']} ({claim['status']})")
print(f"Total: {len(broker_b_claims)} claims")

print("\n✓ RLS working correctly - each broker sees only their claims")

## Step 7: Cleanup (Optional)

Run this cell to delete all AWS resources created in this tutorial.

**Warning**: This will permanently delete:
- Cognito User Pool and users
- PostgreSQL database tables and data (if you want to drop the database)

**Note**: This tutorial uses an existing PostgreSQL database. Only the Cognito resources will be deleted.

In [ ]:
from utils.aws_resources import cleanup_cognito_resources

# Uncomment to cleanup Cognito resources
# cleanup_cognito_resources(
#     region=region,
#     user_pool_name=user_pool_name
# )

# If you want to drop the database tables (optional):
# with conn.cursor() as cur:
#     cur.execute("DROP TABLE IF EXISTS claims CASCADE")
#     cur.execute("DROP TABLE IF EXISTS user_tenants CASCADE")
#     cur.execute("DROP TABLE IF EXISTS tenants CASCADE")
# conn.commit()
# conn.close()
# print("✓ Database tables dropped")